# Discriminative Project – Milestone 1
## Object classification on student-captured photos: from-scratch CNN vs. transfer learning

**Why this notebook is designed the way it is.** The dataset is ~100 photos per Object ID taken by different students. That creates four specific problems, each handled explicitly:

| Problem in the data | How it is handled |
|---|---|
| Objects that are **far away / tiny** in the frame | Images are squashed to 224×224 (never center-cropped, so off-center objects are never cut away); *zoom-out* augmentation; multi-scale test-time augmentation; a dedicated "small object" robustness test |
| Objects that are **cut off / partially occluded** | Random crops + random erasing during training; "cropped" and "occluded" robustness tests |
| **Uneven photo quality** (blur, lighting, low resolution) | TrivialAugment (color, contrast, sharpness, etc.); a "low quality" robustness test |
| **Background shortcut & noisy labels** — each student shot their object on their own backgrounds, and some images are background-only | Out-of-fold label-quality audit to find suspicious images; near-duplicate analysis so test accuracy isn't inflated by burst shots; Grad-CAM to check the model looks at the object, not the room |

**Models compared** (same split, same seed, same evaluation for all):

| ID | Approach | What it tests |
|---|---|---|
| A | Custom plain CNN, trained from scratch | Baseline: how far can we get without pretraining? |
| B | ConvNeXt-Tiny frozen + linear head (linear probe) | Are ImageNet features alone enough? |
| C | ResNet-50 fine-tuned | Classic residual CNN |
| D | EfficientNet-B0 fine-tuned | Small, efficient CNN |
| E | ConvNeXt-Tiny fine-tuned | Modern CNN (expected strongest) |
| F | ConvNeXt-Tiny fine-tuned, *basic* augmentation | Ablation: does the robustness-oriented augmentation help? |

**How to run.** Run the whole notebook top to bottom (VS Code *Run All*, or headless: `jupyter nbconvert --to notebook --execute --ExecutePreprocessor.timeout=-1 --output run_output.ipynb Milestone1_Expert_CNN_Study_LOCAL.ipynb`).

**Crash-safe & resumable — just run it again after any failure, shutdown or timeout:**
- Decoded images, CNN features and the label audit are cached → Batches 0–2 take ~1–2 min on a re-run.
- Every experiment saves a checkpoint after **every epoch**; an interrupted experiment resumes from its last finished epoch.
- Finished experiments are skipped; an experiment whose training finished but evaluation didn't only re-runs the evaluation.
- If one experiment fails, the error is logged (`experiments/<name>/FAILED.txt`) and the next experiment still runs.
- GPU out-of-memory → that experiment automatically restarts with half the batch size.
- Results are stamped with a data + settings signature. If the dataset, split or settings change, old results are moved to `experiments/_stale/` and retrained — never silently reused.
- Logs: `milestone1_outputs/logs/session_*/` (steps.csv, per-batch logs, full log); live training progress: `experiments/<name>/progress.json` and `train.log`.

# BATCH 0 — Setup

In [ ]:
# [B0.01] Config
# ============================ CONFIG — EDIT THESE ============================
PROJECT     = r"C:\Users\csk23\Documents\Zanthosh\Northeastern\SEMESTERs\4th Sem\Deep Learning\PROJECT - 1"
LOCAL_DATA  = PROJECT + r"\Dataset - Updated"       # folder containing images_OBJ001, images_OBJ002, ... (final locked dataset)
DRIVE_DATA  = LOCAL_DATA                             # data is already unzipped locally -> copy step is skipped
OUT_DIR     = PROJECT + r"\milestone1_outputs_v2"    # all results are saved here (v1 kept as backup)

# Images to drop from ALL splits (e.g. background-only shots, confirmed bad images).
# Fill these in after reviewing Batch 1, then re-run from the "Index the dataset" cell.
EXCLUDE_FILES   = set()     # e.g. {"OBJ012_096.jpg", "OBJ007_043.jpg"}
EXCLUDE_INDICES = set()     # shot numbers to drop in EVERY class, e.g. {96, 97, 98, 99, 100}

# Automatic background-only detection (CLIP). Optional: export the course Object List Google Sheet as CSV
# (File → Download → CSV) and save it as object_list.csv in the PROJECT folder — object names make detection more accurate.
AUTO_EXCLUDE_BG  = False
OBJECT_NAMES_CSV = PROJECT + r"\object_list.csv"
OBJECT_NAME_COL  = None     # None = auto-detect; or the exact column header holding object names
BG_THRESHOLD     = 0.5      # flag images whose CLIP "object" probability is below this
MAX_BG_PER_CLASS = 10       # safety cap
KEEP_FILES       = set()    # wrongly flagged object photos to keep, e.g. {"OBJ012_044.jpg"}

CAP_PER_CLASS = None       # None = keep every valid image (TA guideline: do not randomly remove images to equalize folders)

SEED        = 42
IMG_SIZE    = 224
BATCH_SIZE  = 32            # lower to 16 if you get "CUDA out of memory"
NUM_WORKERS = 0             # must be 0 on Windows inside Jupyter (workers can't see notebook-defined classes)
VAL_FRAC, TEST_FRAC = 0.15, 0.15
DUP_THRESHOLD = 0.92        # cosine similarity above which a test image counts as a near-duplicate of a train image
FORCE_RERUN = False         # True = move existing results to experiments/_stale and retrain everything
BALANCE_CLASSES = "auto"    # "auto" = class-weighted loss only if class sizes differ by >15%; True / False to force
CONTINUE_ON_ERROR = True    # a failing experiment is logged and skipped so the others still run (re-run to retry it)
KEEP_AWAKE = True           # stop Windows from sleeping while the notebook runs
# =============================================================================

In [ ]:
# [B0.02] Copy dataset locally (skipped when already local)
import os, shutil, zipfile, time
os.makedirs(OUT_DIR, exist_ok=True)
if not os.path.isdir(LOCAL_DATA) or not os.listdir(LOCAL_DATA):
    t0 = time.time()
    if DRIVE_DATA.lower().endswith(".zip"):
        with zipfile.ZipFile(DRIVE_DATA) as z:
            z.extractall(LOCAL_DATA)
    else:
        shutil.copytree(DRIVE_DATA, LOCAL_DATA, dirs_exist_ok=True)
    print(f"Dataset copied to local disk in {time.time()-t0:.0f}s")
else:
    print("Local copy already present.")

In [ ]:
# [B0.03] Imports, seed, device
import os, re, io, gc, json, math, copy, random, hashlib, shutil, ctypes
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from PIL import Image, ImageOps, ImageFilter, ImageDraw, ImageEnhance
from tqdm.auto import tqdm

import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T, models

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
from scipy.stats import binomtest

RESAMPLE = Image.Resampling
OUT = Path(OUT_DIR); EXP_ROOT = OUT / "experiments"; FIG = OUT / "figures"
for d in (EXP_ROOT, FIG): d.mkdir(parents=True, exist_ok=True)

def set_seed(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed()

device  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = device.type == "cuda"
torch.backends.cudnn.benchmark = True
print("torch", torch.__version__, "| device:", torch.cuda.get_device_name(0) if USE_AMP else "CPU — NVIDIA GPU not detected! See setup notes (Armoury Crate GPU mode / CUDA build of PyTorch)")

# Prevent Windows from sleeping during long training runs (released automatically when the kernel exits)
if KEEP_AWAKE and os.name == "nt":
    ctypes.windll.kernel32.SetThreadExecutionState(0x80000000 | 0x00000001)   # ES_CONTINUOUS | ES_SYSTEM_REQUIRED
    print("Windows sleep is blocked while this notebook runs.")

In [ ]:
# [B0.04] Logging: every step is recorded so errors can be traced back
# Creates OUT_DIR/logs/session_<date_time>/ with:
#   00_environment.txt  versions, GPU, nvidia-smi       config.json   every setting used in this session
#   full_session.log    everything printed, all steps   batch0.log … batch4.log   the same, split by batch
#   steps.csv           one row per executed cell: step, start, end, duration, OK/ERROR + message
#   (each experiment additionally gets experiments/<name>/train.log and run_config.json)
import sys, platform, subprocess, traceback, csv, datetime as _dt

class RunLogger:
    def __init__(self, log_dir):
        self.dir = log_dir; self.dir.mkdir(parents=True, exist_ok=True)
        self.session = open(log_dir / "full_session.log", "a", encoding="utf-8")
        self.batch_files, self.batch, self.extra, self.step = {}, None, [], None
    def targets(self):
        t = [self.session] + self.extra
        if self.batch is not None:
            if self.batch not in self.batch_files:
                self.batch_files[self.batch] = open(self.dir / f"batch{self.batch}.log", "a", encoding="utf-8")
            t.append(self.batch_files[self.batch])
        return t
    def write(self, s):
        for f in self.targets():
            try: f.write(s); f.flush()
            except Exception: pass

class _Tee:
    # Sends everything printed to the notebook AND to the log files
    def __init__(self, stream, logger, skip_progress=False):
        self._stream, self._logger, self._skip = stream, logger, skip_progress
    def write(self, s):
        n = self._stream.write(s)
        if s and not (self._skip and "\r" in s):   # don't flood logs with progress-bar redraws
            self._logger.write(s)
        return n
    def flush(self): self._stream.flush()
    def __getattr__(self, name): return getattr(self._stream, name)

_now = lambda: _dt.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
_ip = get_ipython()
if "_LOG_HANDLERS" in globals():                    # re-running this cell: detach the old logger first
    for ev, fn in _LOG_HANDLERS:
        try: _ip.events.unregister(ev, fn)
        except ValueError: pass
_raw_out = getattr(sys.stdout, "_stream", sys.stdout); _raw_err = getattr(sys.stderr, "_stream", sys.stderr)

SESSION = _dt.datetime.now().strftime("%Y%m%d_%H%M%S")
LOG_DIR = OUT / "logs" / f"session_{SESSION}"
LOGGER = RunLogger(LOG_DIR)
def _attach():
    if not isinstance(sys.stdout, _Tee): sys.stdout = _Tee(_raw_out, LOGGER)
    if not isinstance(sys.stderr, _Tee): sys.stderr = _Tee(_raw_err, LOGGER, skip_progress=True)
_attach()

STEP_RE = re.compile(r"^\s*#\s*\[B(\d+)\.(\d+)\]\s*(.*)")
STEPS_CSV = LOG_DIR / "steps.csv"
if not STEPS_CSV.exists():
    with open(STEPS_CSV, "w", newline="", encoding="utf-8") as fh:
        csv.writer(fh).writerow(["step", "name", "start", "end", "seconds", "status", "message"])

def _pre(info):
    _attach()
    raw = info.raw_cell or ""; m = STEP_RE.match(raw)
    if m: LOGGER.batch = int(m.group(1))
    tag = f"B{m.group(1)}.{m.group(2)}" if m else "untagged"
    name = m.group(3).strip() if m else raw.strip().split("\n")[0][:70]
    LOGGER.step = dict(tag=tag, name=name, start=time.time(), start_s=_now())
    LOGGER.write(f"\n{'=' * 100}\n[{_now()}] START {tag}  {name}\n{'=' * 100}\n")

def _post(result):
    st = LOGGER.step
    if st is None: return
    err = result.error_in_exec or result.error_before_exec
    status, msg = ("OK", "") if err is None else ("ERROR", f"{type(err).__name__}: {err}")
    dur = time.time() - st["start"]
    LOGGER.write(f"[{_now()}] END   {st['tag']}  | {status} {msg} | {dur:.1f}s\n")
    with open(STEPS_CSV, "a", newline="", encoding="utf-8") as fh:
        csv.writer(fh).writerow([st["tag"], st["name"], st["start_s"], _now(), f"{dur:.1f}", status, msg[:500]])
    LOGGER.step = None

def _exc_handler(shell, etype, evalue, tb, tb_offset=None):
    LOGGER.write("\n!!! EXCEPTION — full traceback:\n" + "".join(traceback.format_exception(etype, evalue, tb)))
    shell.showtraceback((etype, evalue, tb), tb_offset=tb_offset)

_ip.events.register("pre_run_cell", _pre); _ip.events.register("post_run_cell", _post)
_ip.set_custom_exc((Exception,), _exc_handler)
_LOG_HANDLERS = [("pre_run_cell", _pre), ("post_run_cell", _post)]

# Environment + config snapshot
env = [f"session      : {SESSION}", f"python       : {sys.version.split()[0]}", f"platform     : {platform.platform()}",
       f"torch        : {torch.__version__} (CUDA build {torch.version.cuda}, cuDNN {torch.backends.cudnn.version()})"]
if USE_AMP:
    pr = torch.cuda.get_device_properties(0); env.append(f"GPU          : {pr.name}, {pr.total_memory / 1e9:.1f} GB")
for mod in ["torchvision", "numpy", "pandas", "sklearn", "scipy", "PIL", "transformers"]:
    try: env.append(f"{mod:<13}: {__import__(mod).__version__}")
    except Exception: env.append(f"{mod:<13}: not installed")
try: env.append("\n" + subprocess.run(["nvidia-smi"], capture_output=True, text=True, timeout=20).stdout)
except Exception as e: env.append(f"nvidia-smi unavailable: {e}")
(LOG_DIR / "00_environment.txt").write_text("\n".join(env), encoding="utf-8")
CONFIG_SNAPSHOT = {k: (sorted(v) if isinstance(v, set) else v) for k, v in globals().items()
                   if k.isupper() and not k.startswith("_") and isinstance(v, (int, float, str, bool, set, type(None)))}
(LOG_DIR / "config.json").write_text(json.dumps(CONFIG_SNAPSHOT, indent=2, default=str), encoding="utf-8")

LOGGER.batch = 0
LOGGER.step = dict(tag="B0.04", name="Logging", start=time.time(), start_s=_now())
print("\n".join(env[:5 + int(USE_AMP)]))
print(f"\n📝 Logging to: {LOG_DIR}")

# BATCH 1 — Data audit (≈3–6 min)
Goal: make sure labels, Object IDs and image files are sound **before** spending GPU time.
At the end of this batch, review the flagged images, update `EXCLUDE_FILES` / `EXCLUDE_INDICES` in the config if needed, and re-run from the next cell.

In [ ]:
# [B1.01] Index the dataset: one folder per Object ID
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def find_class_root(path):
    # Walk down through wrapper folders (e.g. zip -> merged_dataset/) until the class folders are reached.
    path = Path(path)
    while True:
        entries = [e for e in path.iterdir() if not e.name.startswith((".", "__MACOSX"))]
        dirs  = [e for e in entries if e.is_dir()]
        files = [e for e in entries if e.suffix.lower() in IMG_EXTS]
        if len(dirs) == 1 and not files:
            path = dirs[0]
        else:
            return path

DATA_ROOT   = find_class_root(LOCAL_DATA)
class_dirs  = sorted(d for d in DATA_ROOT.iterdir() if d.is_dir() and not d.name.startswith((".", "__")))

def object_id(folder_name):
    # "images_OBJ012" -> "OBJ012" (falls back to the folder name if no OBJ### pattern is found)
    m = re.search(r"OBJ\d+", folder_name, flags=re.IGNORECASE)
    return m.group(0).upper() if m else folder_name

class_names = [object_id(d.name) for d in class_dirs]
assert len(set(class_names)) == len(class_names), "Two folders map to the same Object ID — check folder names!"
NUM_CLASSES = len(class_names)

rows = []
for label, (cls, cdir) in enumerate(zip(class_names, class_dirs)):
    for f in sorted(cdir.iterdir()):
        if f.suffix.lower() not in IMG_EXTS or f.name.startswith("."):
            continue
        m = re.search(r"_(\d+)\.[A-Za-z]+$", f.name)
        rows.append(dict(path=str(f), fname=f.name, class_name=cls, label=label,
                         shot=int(m.group(1)) if m else -1))
df_all = pd.DataFrame(rows)

# Checks
counts = df_all.class_name.value_counts().reindex(class_names)
mismatch = df_all[[not f.upper().startswith(c.upper()) for f, c in zip(df_all.fname, df_all.class_name)]]
dup_names = df_all[df_all.duplicated("fname", keep=False)]

print(f"Root: {DATA_ROOT}")
print(f"{NUM_CLASSES} classes | {len(df_all)} images | per class: min {counts.min()}, median {int(counts.median())}, max {counts.max()}")
if (counts != 100).any():
    print("Classes without exactly 100 images:", counts[counts != 100].to_dict())
if len(mismatch):
    print(f"⚠️ {len(mismatch)} files whose name prefix ≠ folder (check Object IDs!):", mismatch.fname.head(15).tolist())
if len(dup_names):
    print(f"⚠️ {len(dup_names)} files share a filename across folders:", dup_names.path.head(10).tolist())

excluded = df_all.fname.isin(EXCLUDE_FILES) | df_all.shot.isin(EXCLUDE_INDICES)
df = df_all[~excluded].reset_index(drop=True)
print(f"Excluded {int(excluded.sum())} images by config → {len(df)} images kept")

In [ ]:
# [B1.02] Decode every image once into RAM at 224x224 (cached on disk → re-runs take seconds)
# Images are squashed to a square so off-centre objects are never cropped away.
Image.MAX_IMAGE_PIXELS = None   # trusted course dataset; OBJ022 photos are 16128x16128 (PIL blocks >179 MP by default)
CACHE = OUT / "cache"; CACHE.mkdir(exist_ok=True)

def safe_load(path):
    try:
        with Image.open(path) as im:
            im.draft("RGB", (IMG_SIZE * 2, IMG_SIZE * 2))           # fast reduced-size JPEG decode for huge photos
            im = ImageOps.exif_transpose(im).convert("RGB")
            w, h = im.size
            arr = np.asarray(im.resize((IMG_SIZE, IMG_SIZE), RESAMPLE.BICUBIC), dtype=np.uint8)
        with open(path, "rb") as fh:
            md5 = hashlib.md5(fh.read()).hexdigest()
        return arr, w, h, md5
    except Exception:
        return None

def files_signature(paths):
    # Changes if any file is added, removed, replaced or edited, or if IMG_SIZE changes
    h = hashlib.md5(f"img{IMG_SIZE}".encode())
    for p in paths:
        st = os.stat(p); h.update(f"{p}|{st.st_size}|{st.st_mtime_ns}".encode())
    return h.hexdigest()[:16]

_n_listed = len(df)
_sig = files_signature(df.path.values)
_img_cache, _meta_cache = CACHE / f"decoded_{_sig}.npy", CACHE / f"decoded_{_sig}.csv"
if _img_cache.exists() and _meta_cache.exists():
    _meta = pd.read_csv(_meta_cache)
    IMAGES = np.load(_img_cache)
    df = df.merge(_meta, on="path", how="inner")
    assert len(df) == len(IMAGES) == len(_meta) and (df.path.values == _meta.path.values).all(), \
        "Decode cache is inconsistent — delete milestone1_outputs/cache and re-run."
    print(f"Loaded {len(IMAGES)} decoded images from cache ({_img_cache.name}).")
else:
    with ThreadPoolExecutor(4) as ex:
        loaded = list(tqdm(ex.map(safe_load, df.path), total=len(df), desc="decoding"))
    ok = np.array([r is not None for r in loaded])
    if (~ok).any():
        print(f"\u26a0\ufe0f {int((~ok).sum())} unreadable images removed:", df.path[~ok].head(10).tolist())
    loaded = [r for r in loaded if r is not None]
    df = df[ok].reset_index(drop=True)
    IMAGES = np.stack([r[0] for r in loaded])
    df["orig_w"] = [r[1] for r in loaded]; df["orig_h"] = [r[2] for r in loaded]; df["md5"] = [r[3] for r in loaded]
    del loaded
    for old in CACHE.glob("decoded_*"): old.unlink()            # keep only the current cache
    _tmp_img, _tmp_meta = CACHE / f"tmp_{_sig}.npy", CACHE / f"tmp_{_sig}.csv"
    np.save(_tmp_img, IMAGES); df[["path", "orig_w", "orig_h", "md5"]].to_csv(_tmp_meta, index=False)
    os.replace(_tmp_img, _img_cache); os.replace(_tmp_meta, _meta_cache)   # written atomically: never half-saved
    print(f"Decoded images cached → {_img_cache}")

N_UNREADABLE = _n_listed - len(df)
df["row"] = np.arange(len(df))
print(f"IMAGES in RAM: {IMAGES.shape}, {IMAGES.nbytes/1e9:.2f} GB | unreadable files skipped: {N_UNREADABLE}")
print("Most common original sizes:", df.groupby(["orig_w", "orig_h"]).size().sort_values(ascending=False).head(5).to_dict())

dups = df[df.duplicated("md5", keep=False)].groupby("md5")
cross = [g for _, g in dups if g.class_name.nunique() > 1]
print(f"Byte-identical duplicates (pre-bg-removal): {dups.ngroups} groups, {len(cross)} spanning different classes")
for g in cross[:10]: print("   \u26a0\ufe0f", g.path.tolist())

# Stop immediately if any class has zero readable images
missing_cls = [c for c in class_names if not (df.class_name == c).any()]
if missing_cls:
    raise RuntimeError(f"Zero readable images for {missing_cls} — fix the dataset before continuing.")
print(f"All {NUM_CLASSES} classes have at least one readable image. OBJ022: {(df.class_name == 'OBJ022').sum()} images")

### Background image removal by filename
Images whose filename contains `_bg_` or `BG` followed by digits are excluded as background-only shots.

In [ ]:
# [B1.03] Background-only removal by filename
# Match: _bg_ (any case) OR BG followed by digits (any case), e.g. OBJ013_BG001.jpg
import re as _re
_BG_PAT = _re.compile(r'_bg_|bg\d+', _re.IGNORECASE)

df["bg_flag"] = df.fname.apply(lambda f: bool(_BG_PAT.search(f)))

flag = df[df.bg_flag].sort_values(["class_name", "fname"])
per_cls_bg = flag.groupby("class_name").size().reindex(class_names, fill_value=0)
print(f"Background images identified by filename: {len(flag)} total")
zero_bg = per_cls_bg[per_cls_bg == 0].index.tolist()
print(f"Classes with zero bg images ({len(zero_bg)}): {zero_bg}")
print("\nPer-class bg count:")
for cls, n in per_cls_bg.sort_index().items():
    print(f"  {cls}: {n}")

N_BG = len(flag)
flag[["fname", "class_name", "shot"]].to_csv(OUT / "excluded_background_only.csv", index=False)

# Keep bg images in memory for background-shortcut test in Batch 4
BG_IMAGES = IMAGES[flag.row.values].copy()
BG_LABELS = flag.label.values

# Remove from working dataset
keep_mask = np.ones(len(IMAGES), dtype=bool)
keep_mask[flag.row.values] = False
IMAGES = IMAGES[keep_mask]
df = df[~df.bg_flag].reset_index(drop=True)
df["row"] = np.arange(len(df))
print(f"\nRemoved {len(flag)} bg images -> {len(df)} images remain")

In [ ]:
# [B1.04] Remove exact duplicates (before the split), optional per-class cap, re-index

# -- Exact duplicate removal (same MD5, keep first alphabetically) ----------
dup_groups = df[df.duplicated("md5", keep=False)].groupby("md5")
drop_dup, dup_records = [], []
for md5_val, g in dup_groups:
    g_sorted = g.sort_values("fname")
    kept = g_sorted.iloc[0]
    for _, row in g_sorted.iloc[1:].iterrows():
        drop_dup.append(row.name)
        dup_records.append({"removed_fname": row["fname"], "removed_class": row["class_name"],
                             "kept_fname": kept["fname"], "kept_class": kept["class_name"], "md5": md5_val})

pd.DataFrame(dup_records).to_csv(OUT / "removed_duplicates.csv", index=False)
if drop_dup:
    per_dup = pd.DataFrame(dup_records).groupby("removed_class").size()
    print(f"Exact duplicates removed: {len(drop_dup)}")
    print("Per class:", per_dup.to_dict())
    keep_mask = np.ones(len(IMAGES), dtype=bool)
    for idx in drop_dup: keep_mask[idx] = False
    IMAGES = IMAGES[keep_mask]
    df = df.drop(index=drop_dup).reset_index(drop=True)
    df["row"] = np.arange(len(df))
else:
    print("No exact duplicates found.")

# -- Optional cap (OFF by default: CAP_PER_CLASS = None, per the TAs' preprocessing guideline) ---------
rng_cap = np.random.RandomState(SEED)
drop_cap, cap_records = [], []
for cls, g in df.groupby("class_name"):
    if CAP_PER_CLASS and len(g) > CAP_PER_CLASS:
        keep_idx = set(rng_cap.choice(g.index.values, size=CAP_PER_CLASS, replace=False))
        for idx in g.index:
            if idx not in keep_idx:
                drop_cap.append(idx)
                cap_records.append({"fname": df.loc[idx, "fname"], "class_name": cls})

pd.DataFrame(cap_records, columns=["fname", "class_name"]).to_csv(OUT / "capped_images.csv", index=False)
if drop_cap:
    per_cap = pd.DataFrame(cap_records).groupby("class_name").size()
    print(f"\nImages capped to {CAP_PER_CLASS}/class: {len(drop_cap)} dropped")
    print("Per class:", per_cap.to_dict())
    keep_mask = np.ones(len(IMAGES), dtype=bool)
    for idx in drop_cap: keep_mask[idx] = False
    IMAGES = IMAGES[keep_mask]
    df = df.drop(index=drop_cap).reset_index(drop=True)
    df["row"] = np.arange(len(df))
else:
    print("\nNo per-class cap applied — every valid image is kept." if not CAP_PER_CLASS
          else f"\nNo class exceeded {CAP_PER_CLASS} images.")

# -- Final per-class count table ---------------------------------------------
counts = df.class_name.value_counts().reindex(class_names, fill_value=0)
print(f"\nFinal dataset: {len(df)} images | {NUM_CLASSES} classes")
print(f"Per-class: min {counts.min()}, median {int(counts.median())}, max {counts.max()}")
print("\nFull per-class count:")
for cls in class_names:
    print(f"  {cls}: {int(counts[cls])}")


N_DUP, N_CAP = len(drop_dup), len(drop_cap)
DATA_SIG = hashlib.md5("|".join(sorted(df.class_name + "/" + df.md5)).encode()).hexdigest()[:12]
df[["fname", "class_name", "label", "md5"]].to_csv(OUT / "clean_dataset.csv", index=False)
print(f"\nClean dataset signature: {DATA_SIG}  (list saved to clean_dataset.csv)")

In [ ]:
# [B1.05] Look at one class's full shot sequence
# Use this to see whether background-only shots sit at fixed shot numbers (then set EXCLUDE_INDICES).
def show_class(cls, cols=10):
    sub = df[df.class_name == cls].sort_values("shot")
    rows_ = int(np.ceil(len(sub) / cols))
    fig, axes = plt.subplots(rows_, cols, figsize=(cols * 1.3, rows_ * 1.45))
    for ax in np.ravel(axes): ax.axis("off")
    for ax, (_, r) in zip(np.ravel(axes), sub.iterrows()):
        ax.imshow(IMAGES[r.row]); ax.set_title(str(r.shot), fontsize=7)
    fig.suptitle(cls, fontsize=11); plt.tight_layout(); plt.show()

show_class("OBJ012" if "OBJ012" in class_names else class_names[0])
show_class(random.Random(SEED).choice(class_names))

In [ ]:
# [B1.06] Shared transforms, perturbations and dataset
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

class RandomZoomOut:
    # Simulates a farther camera: shrink the photo and paste it onto a heavily blurred copy of itself
    # (a blurred canvas avoids hard black borders the network could latch onto).
    def __init__(self, p=0.25, scale=(0.45, 0.85)):
        self.p, self.scale = p, scale
    def __call__(self, img):
        if random.random() >= self.p:
            return img
        w, h = img.size
        s = random.uniform(*self.scale)
        nw, nh = max(1, int(w * s)), max(1, int(h * s))
        canvas = img.filter(ImageFilter.GaussianBlur(radius=20))
        canvas.paste(img.resize((nw, nh), RESAMPLE.BILINEAR), (random.randint(0, w - nw), random.randint(0, h - nh)))
        return canvas

# Robustness-oriented training augmentation (targets: small/far, cut-off, occluded, varied quality)
robust_train_tf = T.Compose([
    T.RandomResizedCrop(IMG_SIZE, scale=(0.5, 1.0), ratio=(0.8, 1.25)),   # mild: keeps small objects in frame
    RandomZoomOut(p=0.25),
    T.RandomHorizontalFlip(),
    T.TrivialAugmentWide(),                                              # color/contrast/sharpness/geometry
    T.ToTensor(), T.Normalize(MEAN, STD),
    T.RandomErasing(p=0.25, scale=(0.02, 0.15), value="random"),         # occlusion
])
# Basic augmentation (for the ablation experiment F)
basic_train_tf = T.Compose([
    T.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)), T.RandomHorizontalFlip(),
    T.ToTensor(), T.Normalize(MEAN, STD),
])
eval_tf = T.Compose([T.ToTensor(), T.Normalize(MEAN, STD)])

# Deterministic test-time perturbations for the robustness evaluation (same perturbation per image for every model)
ROBUSTNESS_KINDS = ["small_object", "cropped", "occluded", "low_quality"]
def perturb(img, kind, rng):
    w, h = img.size
    if kind == "small_object":     # object 2x farther away
        canvas = img.filter(ImageFilter.GaussianBlur(radius=20))
        small = img.resize((w // 2, h // 2), RESAMPLE.BILINEAR)
        canvas.paste(small, ((w - small.width) // 2, (h - small.height) // 2))
        return canvas
    if kind == "cropped":          # only a random 70%×70% window is visible
        cw, ch = int(w * 0.7), int(h * 0.7)
        x, y = rng.randint(0, w - cw + 1), rng.randint(0, h - ch + 1)
        return img.crop((x, y, x + cw, y + ch)).resize((w, h), RESAMPLE.BICUBIC)
    if kind == "occluded":         # gray box over 25% of the image
        img = img.copy(); bw, bh = w // 2, h // 2
        x, y = rng.randint(0, w - bw + 1), rng.randint(0, h - bh + 1)
        ImageDraw.Draw(img).rectangle([x, y, x + bw, y + bh], fill=(124, 116, 104))
        return img
    if kind == "low_quality":      # 4x lower resolution + dim lighting + heavy JPEG compression
        img = img.resize((w // 4, h // 4), RESAMPLE.BILINEAR).resize((w, h), RESAMPLE.BILINEAR)
        img = ImageEnhance.Brightness(img).enhance(0.6)
        buf = io.BytesIO(); img.save(buf, "JPEG", quality=20); buf.seek(0)
        return Image.open(buf).convert("RGB")
    raise ValueError(kind)

class ArrayDataset(Dataset):
    def __init__(self, rows, labels, tf, perturb_kind="clean"):
        self.rows, self.labels, self.tf, self.kind = np.asarray(rows), np.asarray(labels), tf, perturb_kind
    def __len__(self): return len(self.rows)
    def __getitem__(self, i):
        r = int(self.rows[i])
        img = Image.fromarray(IMAGES[r])
        if self.kind != "clean":
            img = perturb(img, self.kind, np.random.RandomState(SEED * 100003 + r))
        return self.tf(img), int(self.labels[i])

def make_loader(frame, tf, shuffle=False, drop_last=False, perturb_kind="clean", bs=None):
    g = torch.Generator(); g.manual_seed(SEED)
    return DataLoader(ArrayDataset(frame["row"].values, frame["label"].values, tf, perturb_kind),
                      batch_size=bs or BATCH_SIZE, shuffle=shuffle, drop_last=drop_last,
                      num_workers=NUM_WORKERS, pin_memory=USE_AMP, generator=g)

# Preview the robustness perturbations on one image
demo = Image.fromarray(IMAGES[0])
fig, axes = plt.subplots(1, 5, figsize=(13, 3))
for ax, k in zip(axes, ["clean"] + ROBUSTNESS_KINDS):
    ax.imshow(demo if k == "clean" else perturb(demo, k, np.random.RandomState(0))); ax.set_title(k); ax.axis("off")
plt.tight_layout(); plt.savefig(FIG / "robustness_perturbations.png", dpi=150); plt.show()

In [ ]:
# [B1.07] Frozen ConvNeXt-Tiny features for every image (used for the label audit and near-duplicate check)
feat_file = OUT / "convnext_features.npz"
if feat_file.exists() and np.array_equal(np.load(feat_file, allow_pickle=True)["paths"], df.path.values):
    FEATS = np.load(feat_file, allow_pickle=True)["feats"]
    print("Loaded cached features:", FEATS.shape)
else:
    fx = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1)
    fx.classifier[2] = nn.Identity()
    fx = fx.eval().to(device)
    feats = []
    with torch.no_grad():
        for x, _ in tqdm(make_loader(df, eval_tf), desc="features"):
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
                feats.append(fx(x.to(device)).float().cpu())
    FEATS = torch.cat(feats).numpy()
    np.savez(feat_file, feats=FEATS, paths=df.path.values)
    del fx; torch.cuda.empty_cache()
    print("Extracted features:", FEATS.shape)
FEATS_N = FEATS / np.linalg.norm(FEATS, axis=1, keepdims=True)

In [ ]:
# [B1.08] Label-quality audit: out-of-fold predictions from a linear classifier on frozen features
# Each image is scored by a model that never saw it. A low probability for its own label means it is
# mislabeled, in the wrong Object ID folder, background-only, or extremely hard (tiny/blurred/cut off).
oof_file = OUT / "label_audit_oof.npz"
y_all = df.label.values

# Handle non-contiguous labels (e.g. a class that lost all images creates a gap in label indices)
unique_labels = np.unique(y_all)
label_to_oof_col = {int(l): i for i, l in enumerate(unique_labels)}
y_contiguous = np.array([label_to_oof_col[int(l)] for l in y_all])

if oof_file.exists() and np.array_equal(np.load(oof_file, allow_pickle=True)["paths"], df.path.values):
    OOF = np.load(oof_file, allow_pickle=True)["oof"]
else:
    clf = make_pipeline(StandardScaler(), LogisticRegression(C=0.5, max_iter=2000))
    OOF = cross_val_predict(clf, FEATS, y_contiguous, cv=StratifiedKFold(5, shuffle=True, random_state=SEED),
                            method="predict_proba", n_jobs=-1)
    np.savez(oof_file, oof=OOF, paths=df.path.values)

df["self_conf"] = OOF[np.arange(len(df)), y_contiguous]
df["oof_pred"]  = unique_labels[OOF.argmax(1)]   # map oof-column index back to original label
df["oof_pred_name"] = [class_names[i] for i in df.oof_pred]
print(f"Out-of-fold linear-probe accuracy (5-fold CV): {(df.oof_pred == df.label).mean():.4f}")

per_class = df.assign(ok=df.oof_pred == df.label).groupby("class_name").ok.mean().sort_values()
print("\nHardest classes (OOF accuracy):", per_class.head(8).round(3).to_dict())

suspects = df.sort_values("self_conf").head(100)
suspects[["fname", "class_name", "oof_pred_name", "self_conf", "shot"]].to_csv(OUT / "label_audit_suspects.csv", index=False)

# Strongly confident disagreements = likely wrong folder / wrong Object ID
wrong_folder = df[(df.oof_pred != df.label) & (OOF.max(1) > 0.8)]
print(f"\nConfidently assigned to ANOTHER class (p>0.8): {len(wrong_folder)} images")
print(wrong_folder[["fname", "class_name", "oof_pred_name"]].head(20).to_string(index=False))


In [ ]:
# [B1.09] Review the 40 most suspicious images
s = suspects.head(40)
fig, axes = plt.subplots(5, 8, figsize=(16, 11))
for ax in axes.ravel(): ax.axis("off")
for ax, (_, r) in zip(axes.ravel(), s.iterrows()):
    ax.imshow(IMAGES[r.row])
    ax.set_title(f"{r.fname}\npred {r.oof_pred_name}  p(own)={r.self_conf:.2f}", fontsize=7)
plt.suptitle("Lowest out-of-fold confidence for their own label", fontsize=12)
plt.tight_layout(); plt.savefig(FIG / "label_audit_suspects.png", dpi=150); plt.show()
print("Full list: label_audit_suspects.csv in OUT_DIR. Add clear background-only / wrong images to EXCLUDE_FILES.")

### ✋ Review after Batch 1 (informational — the notebook continues automatically) (≈2 minutes of looking, no manual searching)
- **Background-only shots are removed automatically.** Just glance at the *FLAGGED* grid: if a real object photo got flagged, add its filename to `KEEP_FILES`. If an obvious empty scene appears in the *Borderline, KEPT* grid, add it to `EXCLUDE_FILES`. If everything looks right, change nothing.
- The *suspects* grid (lowest label confidence) catches other problems — wrong folder, wrong Object ID. Only act on clear errors; do **not** remove images just because they are hard (tiny, blurred, cut off).
- If you changed anything in the config, re-run from **"Index the dataset"**. Then continue to Batch 2.

# BATCH 2 — Split, leakage check, models, training engine (≈1 min)

In [ ]:
# [B2.01] Stratified 70/15/15 split, fixed seed, identical for every model
# stratify=label → each Object ID is split 70/15/15 on its own, so every class appears in train, val and test
# in the same proportion (e.g. 95 images → 66 / 14 / 15).
idx = np.arange(len(df))
tr_idx, tmp_idx = train_test_split(idx, test_size=VAL_FRAC + TEST_FRAC, stratify=df.label, random_state=SEED)
va_idx, te_idx  = train_test_split(tmp_idx, test_size=TEST_FRAC / (VAL_FRAC + TEST_FRAC),
                                   stratify=df.label.values[tmp_idx], random_state=SEED)
df["split"] = ""
df.loc[tr_idx, "split"] = "train"; df.loc[va_idx, "split"] = "val"; df.loc[te_idx, "split"] = "test"
train_df = df[df.split == "train"].reset_index(drop=True)
val_df   = df[df.split == "val"].reset_index(drop=True)
test_df  = df[df.split == "test"].reset_index(drop=True)
df[["fname", "class_name", "label", "split"]].to_csv(OUT / "split.csv", index=False)
print(f"train {len(train_df)} | val {len(val_df)} | test {len(test_df)} | classes {NUM_CLASSES}")

# Data signature: changes if the cleaned dataset or the split changes → stale results can never be reused
SPLIT_SIG = hashlib.md5("|".join(sorted(f"{a}/{b}/{c_}" for a, b, c_ in zip(df.class_name, df.md5, df.split))).encode()).hexdigest()[:12]
manifest = dict(data_sig=DATA_SIG, split_sig=SPLIT_SIG, images=len(df), classes=NUM_CLASSES,
                train=len(train_df), val=len(val_df), test=len(test_df),
                removed=dict(unreadable=N_UNREADABLE, background=N_BG, duplicates=N_DUP, capped=N_CAP), created=_now())
(OUT / "data_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print("Split signature:", SPLIT_SIG, "| manifest saved to data_manifest.json")

In [ ]:
# [B2.02] Verify every Object ID is split in the same 70/15/15 proportion; handle unequal class sizes
tab = (df.groupby(["class_name", "split"]).size().unstack(fill_value=0)
         .reindex(index=class_names, columns=["train", "val", "test"], fill_value=0))
tab["total"] = tab.sum(axis=1)
for s_ in ["train", "val", "test"]: tab[f"{s_}_%"] = (100 * tab[s_] / tab.total).round(1)
tab.to_csv(OUT / "split_per_class.csv")
assert (tab[["train", "val", "test"]] > 0).all().all(), "Some Object ID is missing from a split!"
print("Per-class split (first 10 shown, full table in split_per_class.csv):"); display(tab.head(10))
print(f"train % per class: {tab['train_%'].min()}–{tab['train_%'].max()} | "
      f"val %: {tab['val_%'].min()}–{tab['val_%'].max()} | test %: {tab['test_%'].min()}–{tab['test_%'].max()}")

counts_tr = tab["train"].values.astype(float)
imbalance = counts_tr.max() / counts_tr.min()
use_w = BALANCE_CLASSES is True or (BALANCE_CLASSES == "auto" and imbalance > 1.15)
CLASS_WEIGHTS = (torch.tensor(counts_tr.sum() / (NUM_CLASSES * counts_tr), dtype=torch.float32, device=device)
                 if use_w else None)
print(f"Training images per class: min {int(counts_tr.min())}, max {int(counts_tr.max())} (ratio {imbalance:.2f}) → "
      + ("class-weighted loss ENABLED (smaller classes count more, so no object dominates)." if use_w
         else "classes are balanced enough → standard loss."))

In [ ]:
# [B2.03] Near-duplicate leakage check
# Students often take near-identical consecutive shots. If one lands in train and its twin in test,
# test accuracy overstates generalization. We flag test images whose most similar SAME-CLASS train image
# exceeds DUP_THRESHOLD, and later report accuracy on the remaining "hard" test subset as well.
S = FEATS_N[test_df.row.values] @ FEATS_N[train_df.row.values].T
S[test_df.label.values[:, None] != train_df.label.values[None, :]] = -1
test_df["max_train_sim"] = S.max(1)
test_df["nn_train_row"]  = train_df.row.values[S.argmax(1)]
test_df["near_dup"]      = test_df.max_train_sim >= DUP_THRESHOLD
print(f"Near-duplicate test images (sim ≥ {DUP_THRESHOLD}): {test_df.near_dup.sum()} / {len(test_df)} "
      f"({test_df.near_dup.mean():.1%}) → 'hard' subset = {(~test_df.near_dup).sum()} images")

fig = plt.figure(figsize=(14, 3.5))
ax = fig.add_subplot(1, 4, 1)
ax.hist(test_df.max_train_sim, bins=40); ax.axvline(DUP_THRESHOLD, c="r", ls="--")
ax.set_title("Test → nearest train similarity"); ax.set_xlabel("cosine similarity")
top = test_df.sort_values("max_train_sim", ascending=False).head(3)
for k, (_, r) in enumerate(top.iterrows()):
    a = fig.add_subplot(1, 4, k + 2)
    a.imshow(np.concatenate([IMAGES[r.row], IMAGES[r.nn_train_row]], axis=1)); a.axis("off")
    a.set_title(f"test | train   sim={r.max_train_sim:.3f}", fontsize=8)
plt.tight_layout(); plt.savefig(FIG / "near_duplicates.png", dpi=150); plt.show()

In [ ]:
# [B2.04] Models
class CustomCNN(nn.Module):
    # Plain VGG-style CNN (no skip connections), trained from scratch. 5 stages: 224 -> 112 -> 56 -> 28 -> 14 -> 7.
    #  • padding=1 on every 3x3 conv ("same" padding): spatial size is kept inside a stage, so objects at the
    #    image border are not eroded away — important for cut-off / off-center objects.
    #  • BatchNorm after every conv (bias=False because BN has its own shift): stabilises and speeds up training
    #    from scratch and acts as a mild regulariser.
    #  • MaxPool 2x2 between stages: halves resolution, doubles receptive field, adds small-shift tolerance.
    #  • Global average pooling instead of flatten: 384 features instead of 384*7*7 (≈ 19k), far fewer parameters,
    #    less overfitting, and the object can be anywhere in the frame (helps far / off-center objects).
    #  • Dropout before the classifier; Kaiming init matched to ReLU.
    def __init__(self, n, widths=(32, 64, 128, 256, 384), dropout=0.3):
        super().__init__()
        stages, cin = [], 3
        for w in widths:
            stages.append(nn.Sequential(
                nn.Conv2d(cin, w, 3, padding=1, bias=False), nn.BatchNorm2d(w), nn.ReLU(inplace=True),
                nn.Conv2d(w, w, 3, padding=1, bias=False),   nn.BatchNorm2d(w), nn.ReLU(inplace=True),
                nn.MaxPool2d(2)))
            cin = w
        self.features = nn.Sequential(*stages)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(nn.Flatten(), nn.Dropout(dropout), nn.Linear(cin, n))
        for mod in self.modules():
            if isinstance(mod, nn.Conv2d):
                nn.init.kaiming_normal_(mod.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(mod, nn.BatchNorm2d):
                nn.init.ones_(mod.weight); nn.init.zeros_(mod.bias)
            elif isinstance(mod, nn.Linear):
                nn.init.normal_(mod.weight, 0, 0.01); nn.init.zeros_(mod.bias)
    def forward(self, x):
        return self.classifier(self.pool(self.features(x)))

class LinearProbe(nn.Module):
    # Frozen ImageNet ConvNeXt-Tiny + one trainable linear layer.
    def __init__(self, n):
        super().__init__()
        self.backbone = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1)
        self.backbone.classifier[2] = nn.Identity()
        for p in self.backbone.parameters():
            p.requires_grad = False
        self.head = nn.Linear(768, n)
    def train(self, mode=True):
        super().train(mode); self.backbone.eval()   # keep frozen backbone deterministic (no stochastic depth)
        return self
    def forward(self, x):
        return self.head(self.backbone(x))

def build_model(arch, n):
    # Returns (model, name prefix of the newly initialised head)
    if arch == "custom_cnn":
        return CustomCNN(n), "classifier"
    if arch == "linear_probe":
        return LinearProbe(n), "head"
    if arch == "resnet50":
        m = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        m.fc = nn.Linear(m.fc.in_features, n); return m, "fc"
    if arch == "efficientnet_b0":
        m = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, n); return m, "classifier"
    if arch == "convnext_tiny":
        m = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1)
        m.classifier[2] = nn.Linear(m.classifier[2].in_features, n); return m, "classifier"
    raise ValueError(arch)

GRADCAM_LAYER = {
    "custom_cnn":      lambda m: m.features[-1],
    "linear_probe":    lambda m: m.backbone.features[-1],
    "resnet50":        lambda m: m.layer4,
    "efficientnet_b0": lambda m: m.features[-1],
    "convnext_tiny":   lambda m: m.features[-1],
}

In [ ]:
# [B2.05] Architecture check: layer shapes, normalisation and pooling of every model (for the report)
m = CustomCNN(NUM_CLASSES).eval(); x = torch.zeros(1, 3, IMG_SIZE, IMG_SIZE)
print("Custom CNN — shape after each stage (padding keeps size inside a stage, max-pool halves it):")
print(f"  {'input':<26}{str(tuple(x.shape)):<22}")
with torch.no_grad():
    for k, stage in enumerate(m.features):
        x = stage(x)
        print(f"  {'stage ' + str(k + 1) + ' (2×[conv-BN-ReLU] + pool)':<26}{str(tuple(x.shape)):<22}"
              f"{sum(p.numel() for p in stage.parameters()):>10,} params")
    x = m.pool(x); print(f"  {'global avg pool':<26}{str(tuple(x.shape)):<22}")
    x = m.classifier(x); print(f"  {'dropout + linear':<26}{str(tuple(x.shape)):<22}"
                                f"{sum(p.numel() for p in m.classifier.parameters()):>10,} params")

arch_rows = []
for arch in ["custom_cnn", "linear_probe", "resnet50", "efficientnet_b0", "convnext_tiny"]:
    mm, _ = build_model(arch, NUM_CLASSES)
    kinds = [type(z).__name__ for z in mm.modules()]
    arch_rows.append({"model": arch,
                      "params_M": round(sum(p.numel() for p in mm.parameters()) / 1e6, 2),
                      "trainable_M": round(sum(p.numel() for p in mm.parameters() if p.requires_grad) / 1e6, 2),
                      "BatchNorm layers": kinds.count("BatchNorm2d"),
                      "LayerNorm layers": kinds.count("LayerNorm") + kinds.count("LayerNorm2d"),
                      "pooling": "global avg" if any(k in kinds for k in ["AdaptiveAvgPool2d"]) else "-"})
    del mm
arch_df = pd.DataFrame(arch_rows); arch_df.to_csv(OUT / "architectures.csv", index=False)
print(); display(arch_df)
print("Note: ResNet-50 and EfficientNet-B0 use BatchNorm; ConvNeXt uses LayerNorm by design. "
      "Pretrained architectures are kept exactly as designed — changing their layers would break the pretrained weights.")

In [ ]:
# [B2.06] Training engine — checkpointed every epoch, resumable, fault-tolerant
# What happens when an experiment cell runs:
#   metrics.json present             → finished earlier: skipped instantly
#   trained.json + best.pt present   → training finished earlier: only the test evaluation is (re)done
#   checkpoint.pt present            → interrupted earlier: training resumes after the last completed epoch
#   nothing                          → trained from scratch
# Every saved file carries SPLIT_SIG (data) and a config signature. If either changed, the old files are moved to
# experiments/_stale/ and the experiment is retrained. All writes are atomic (temp file + rename): a crash
# during saving can never leave a corrupted checkpoint behind.
HIST_HEADER = "epoch,train_loss,train_acc,val_loss,val_acc,val_macro_f1,lr,sec,gpu_mem_gb\n"

def atomic_torch_save(obj, path):
    tmp = path.with_name(path.name + ".tmp"); torch.save(obj, tmp); os.replace(tmp, path)

def atomic_write(path, text):
    tmp = path.with_name(path.name + ".tmp"); tmp.write_text(text, encoding="utf-8"); os.replace(tmp, path)

def cfg_signature(cfg):
    key = {k: v for k, v in cfg.items() if k != "desc"}
    key.update(img_size=IMG_SIZE, class_weighted=CLASS_WEIGHTS is not None, seed=SEED)
    return hashlib.md5(json.dumps(key, sort_keys=True, default=str).encode()).hexdigest()[:12]

def archive_experiment(exp_dir, reason):
    dst = EXP_ROOT / "_stale" / f"{exp_dir.name}_{_dt.datetime.now():%Y%m%d_%H%M%S}"
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.move(str(exp_dir), str(dst)); exp_dir.mkdir(parents=True, exist_ok=True)
    print(f"\u26a0\ufe0f {exp_dir.name}: {reason} → previous files moved to {dst}")

def experiment_status(name):
    d = EXP_ROOT / name
    if (d / "metrics.json").exists():    return "done"
    if (d / "trained.json").exists():    return "trained, evaluation pending"
    if (d / "checkpoint.pt").exists():   return "interrupted (resumable)"
    if (d / "FAILED.txt").exists():      return "failed"
    return "not started"

def param_groups(model, head_prefix, lr_head, lr_backbone, wd):
    # Discriminative learning rates (new head learns faster than pretrained backbone); no weight decay on biases/norms
    groups = {}
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        groups.setdefault((name.startswith(head_prefix), p.ndim > 1), []).append(p)
    return [{"params": ps, "lr": lr_head if is_head else lr_backbone, "weight_decay": wd if decay else 0.0}
            for (is_head, decay), ps in groups.items()]

def to_dev(x):
    return x.to(device, non_blocking=True).to(memory_format=torch.channels_last)

@torch.no_grad()
def predict_probs(model, loader, tta=False):
    # Softmax probabilities. TTA = average of original, horizontal flip and 1.25x center zoom (helps tiny objects).
    model.eval(); out = []
    for x, _ in loader:
        x = to_dev(x)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
            p = model(x).float().softmax(1)
            if tta:
                p = p + model(torch.flip(x, dims=[3])).float().softmax(1)
                s = int(IMG_SIZE * 0.8); o = (IMG_SIZE - s) // 2
                z = F.interpolate(x[:, :, o:o + s, o:o + s], size=(IMG_SIZE, IMG_SIZE), mode="bilinear", align_corners=False)
                p = (p + model(z).float().softmax(1)) / 3
        out.append(p.cpu())
    return torch.cat(out).numpy()

def fit(model, head_prefix, cfg, exp_dir, bs, csig):
    tf = robust_train_tf if cfg["aug"] == "robust" else basic_train_tf
    train_loader = make_loader(train_df, tf, shuffle=True, drop_last=True, bs=bs)
    val_loader = make_loader(val_df, eval_tf, bs=bs)
    opt = torch.optim.AdamW(param_groups(model, head_prefix, cfg["lr_head"], cfg["lr_backbone"], cfg["wd"]))
    spe = len(train_loader); total = cfg["epochs"] * spe; warm = max(1, cfg["warmup_epochs"] * spe)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s + 1) / warm if s < warm else
                                              0.5 * (1 + math.cos(math.pi * min(1.0, (s - warm) / max(1, total - warm)))))
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    crit = nn.CrossEntropyLoss(weight=CLASS_WEIGHTS, label_smoothing=cfg["label_smoothing"])
    y_val = val_loader.dataset.labels
    ckpt_path, prog_path, hist_path = exp_dir / "checkpoint.pt", exp_dir / "progress.json", exp_dir / "history.csv"

    st = dict(epoch=0, best_f1=-1.0, best_loss=float("inf"), best_epoch=0, wait=0, stopped=False, train_sec=0.0, history=[])
    best_state = None
    if ckpt_path.exists():
        ck = torch.load(ckpt_path, map_location="cpu", weights_only=False)
        if ck.get("data_sig") == SPLIT_SIG and ck.get("cfg_sig") == csig and ck.get("batch_size") == bs:
            model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"])
            sched.load_state_dict(ck["sched"]); scaler.load_state_dict(ck["scaler"])
            st, best_state = ck["state"], ck["best_state"]
            print(f"\u21bb Resuming from checkpoint: {st['epoch']} epoch(s) already done, "
                  f"best val-F1 {st['best_f1']:.4f} at epoch {st['best_epoch']}")
        else:
            print("Checkpoint belongs to different data/settings → starting this experiment fresh")
            ckpt_path.unlink()
        del ck
    with open(hist_path, "w", encoding="utf-8") as fh:            # rebuilt from the checkpoint → never duplicate epochs
        fh.write(HIST_HEADER); fh.writelines(st["history"])

    for ep in range(st["epoch"] + 1, cfg["epochs"] + 1):
        if st["stopped"]:
            break
        set_seed(SEED * 1000 + ep); train_loader.generator.manual_seed(SEED * 1000 + ep)   # same order & augmentations even after a resume
        t0 = time.time(); model.train()
        if USE_AMP: torch.cuda.reset_peak_memory_stats()
        tot, correct, n = 0.0, 0, 0
        for x, y in train_loader:
            x, y = to_dev(x), y.to(device, non_blocking=True)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
                out = model(x); loss = crit(out, y)
            if not torch.isfinite(loss):
                raise FloatingPointError(f"Loss became {loss.item()} at epoch {ep} — training diverged")
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step()
            tot += loss.item() * x.size(0); correct += (out.argmax(1) == y).sum().item(); n += x.size(0)

        pv = predict_probs(model, val_loader)
        v_loss = float(-np.log(np.clip(pv[np.arange(len(y_val)), y_val], 1e-8, None)).mean())
        v_pred = pv.argmax(1)
        v_acc, v_f1 = accuracy_score(y_val, v_pred), f1_score(y_val, v_pred, average="macro")
        sec = time.time() - t0
        mem = torch.cuda.max_memory_allocated() / 1e9 if USE_AMP else 0.0
        row = (f"{ep},{tot/n:.5f},{correct/n:.5f},{v_loss:.5f},{v_acc:.5f},{v_f1:.5f},"
               f"{opt.param_groups[0]['lr']:.2e},{sec:.1f},{mem:.2f}\n")
        st["history"].append(row)
        with open(hist_path, "a", encoding="utf-8") as fh: fh.write(row)
        print(f"  ep {ep:02d}/{cfg['epochs']} | train loss {tot/n:.3f} acc {correct/n:.3f} | "
              f"val loss {v_loss:.3f} acc {v_acc:.3f} F1 {v_f1:.3f} | {sec:.0f}s | GPU {mem:.1f} GB")

        if v_f1 > st["best_f1"] + 1e-6 or (abs(v_f1 - st["best_f1"]) <= 1e-6 and v_loss < st["best_loss"]):
            st.update(best_f1=float(v_f1), best_loss=v_loss, best_epoch=ep, wait=0)
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            st["wait"] += 1
        st["epoch"] = ep; st["train_sec"] += sec
        st["stopped"] = st["wait"] >= cfg["patience"]
        atomic_torch_save(dict(model=model.state_dict(), opt=opt.state_dict(), sched=sched.state_dict(),
                               scaler=scaler.state_dict(), best_state=best_state, state=st,
                               data_sig=SPLIT_SIG, cfg_sig=csig, batch_size=bs), ckpt_path)
        atomic_write(prog_path, json.dumps(dict(epoch=ep, max_epochs=cfg["epochs"], best_val_macro_f1=st["best_f1"],
                                                best_epoch=st["best_epoch"], batch_size=bs, updated=_now()), indent=2))
        if st["stopped"]:
            print(f"  early stop (no val-F1 gain for {cfg['patience']} epochs)")

    model.load_state_dict(best_state)
    return dict(best_val_macro_f1=float(st["best_f1"]), best_val_loss=float(st["best_loss"]),
                best_epoch=st["best_epoch"], epochs_run=st["epoch"], train_min=st["train_sec"] / 60)

def evaluate_experiment(model, name, cfg, exp_dir, info, bs, csig):
    test_loader = make_loader(test_df, eval_tf, bs=bs)
    if USE_AMP: torch.cuda.synchronize()
    t1 = time.time(); probs = {"clean": predict_probs(model, test_loader)}
    if USE_AMP: torch.cuda.synchronize()
    ms_img = (time.time() - t1) / len(test_df) * 1000
    probs["tta"] = predict_probs(model, test_loader, tta=True)
    for k in ROBUSTNESS_KINDS:
        probs[k] = predict_probs(model, make_loader(test_df, eval_tf, perturb_kind=k, bs=bs))
    y = test_df.label.values
    tmp = exp_dir / "test_probs.tmp.npz"
    np.savez_compressed(tmp, y=y, **probs); os.replace(tmp, exp_dir / "test_probs.npz")
    acc = lambda p: float(accuracy_score(y, p.argmax(1)))
    metrics = dict(name=name, arch=cfg["arch"], aug=cfg["aug"], desc=cfg["desc"],
                   params_M=sum(p.numel() for p in model.parameters()) / 1e6,
                   trainable_M=sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6,
                   ms_per_img=ms_img, **info,
                   test_acc=acc(probs["clean"]),
                   test_macro_f1=float(f1_score(y, probs["clean"].argmax(1), average="macro")),
                   test_acc_tta=acc(probs["tta"]),
                   **{f"acc_{k}": acc(probs[k]) for k in ROBUSTNESS_KINDS},
                   batch_size=bs, data_sig=SPLIT_SIG, cfg_sig=csig, finished=_now())
    atomic_write(exp_dir / "metrics.json", json.dumps(metrics, indent=2))     # written last = "done" marker
    print(f"\u2192 {name}: test acc {metrics['test_acc']:.4f} | macro-F1 {metrics['test_macro_f1']:.4f} | "
          f"TTA {metrics['test_acc_tta']:.4f} | trained {info['train_min']:.1f} min")
    return metrics

def run_experiment(name):
    cfg = {**DEFAULTS, **EXPERIMENTS[name]}
    csig = cfg_signature(cfg)
    exp_dir = EXP_ROOT / name; exp_dir.mkdir(parents=True, exist_ok=True)

    # Never reuse results produced with different data or settings
    for marker in ("metrics.json", "trained.json"):
        p = exp_dir / marker
        if p.exists():
            meta = json.loads(p.read_text(encoding="utf-8"))
            if FORCE_RERUN:                             archive_experiment(exp_dir, "FORCE_RERUN=True"); break
            if meta.get("data_sig") != SPLIT_SIG:        archive_experiment(exp_dir, "dataset/split changed since training"); break
            if meta.get("cfg_sig") != csig:              archive_experiment(exp_dir, "experiment settings changed"); break
    if (exp_dir / "metrics.json").exists():
        m = json.loads((exp_dir / "metrics.json").read_text(encoding="utf-8"))
        print(f"\u2713 {name} already finished — skipped (test acc {m['test_acc']:.4f}, macro-F1 {m['test_macro_f1']:.4f})")
        return m

    exp_log = open(exp_dir / "train.log", "a", encoding="utf-8")
    LOGGER.extra.append(exp_log)                      # everything printed below also goes to train.log
    model = None
    try:
        print(f"\n========== {name}: {cfg['desc']} | status: {experiment_status(name)} | {_now()} ==========")
        (exp_dir / "FAILED.txt").unlink(missing_ok=True)
        trained = exp_dir / "trained.json"
        if trained.exists():
            tinfo = json.loads(trained.read_text(encoding="utf-8")); bs, info = tinfo["batch_size"], tinfo["fit"]
            print("Training finished in an earlier run → running the test evaluation only")
            model, _ = build_model(cfg["arch"], NUM_CLASSES)
            model.load_state_dict(torch.load(exp_dir / "best.pt", map_location="cpu"))
            model = model.to(device, memory_format=torch.channels_last)
        else:
            bs = BATCH_SIZE
            prog = exp_dir / "progress.json"
            if (exp_dir / "checkpoint.pt").exists() and prog.exists():
                bs = json.loads(prog.read_text(encoding="utf-8")).get("batch_size", BATCH_SIZE)   # resume with the batch size it used
            run_cfg = {**cfg, "class_weighted_loss": CLASS_WEIGHTS is not None, "batch_size": bs, "img_size": IMG_SIZE,
                       "seed": SEED, "session": SESSION, "data_sig": SPLIT_SIG, "cfg_sig": csig, "started": _now()}
            atomic_write(exp_dir / "run_config.json", json.dumps(run_cfg, indent=2))
            print("config:", json.dumps(run_cfg))
            while True:
                oom = False
                try:
                    set_seed(SEED)
                    model, head_prefix = build_model(cfg["arch"], NUM_CLASSES)
                    model = model.to(device, memory_format=torch.channels_last)
                    info = fit(model, head_prefix, cfg, exp_dir, bs, csig)
                    break
                except torch.cuda.OutOfMemoryError:
                    oom = True
                if oom:                                   # cleanup outside the except block so GPU memory is really freed
                    model = None; gc.collect(); torch.cuda.empty_cache()
                    for f_ in ("checkpoint.pt", "progress.json"): (exp_dir / f_).unlink(missing_ok=True)
                    if bs <= 8:
                        raise RuntimeError("GPU out of memory even at batch size 8")
                    bs //= 2
                    print(f"\u26a0\ufe0f GPU out of memory → restarting {name} from scratch with batch size {bs}")
            atomic_torch_save(model.state_dict(), exp_dir / "best.pt")
            atomic_write(trained, json.dumps(dict(fit=info, batch_size=bs, data_sig=SPLIT_SIG, cfg_sig=csig,
                                                  finished=_now()), indent=2))
            (exp_dir / "checkpoint.pt").unlink(missing_ok=True)    # best.pt + trained.json now cover it
        return evaluate_experiment(model, name, cfg, exp_dir, info, bs, csig)
    except Exception:
        tb = traceback.format_exc()
        atomic_write(exp_dir / "FAILED.txt", f"{_now()}\n{tb}")
        print(f"!!! {name} FAILED:\n{tb}")
        if CONTINUE_ON_ERROR:
            print(f"\u2192 continuing with the next experiment. Re-run the notebook later to retry {name} "
                  f"(it resumes from its last completed epoch).")
            return None
        raise
    finally:
        LOGGER.extra.remove(exp_log); exp_log.close()
        model = None; gc.collect()
        if USE_AMP: torch.cuda.empty_cache()

DEFAULTS = dict(aug="robust", epochs=20, warmup_epochs=1, patience=6, label_smoothing=0.1)
EXPERIMENTS = {
    "A_custom_cnn_scratch":   dict(arch="custom_cnn", epochs=60, warmup_epochs=3, patience=15,
                                   lr_head=2e-3, lr_backbone=2e-3, wd=0.05,
                                   desc="Plain 10-layer CNN trained from scratch"),
    "B_convnext_linear_probe": dict(arch="linear_probe", epochs=12, patience=5,
                                   lr_head=2e-3, lr_backbone=0.0, wd=1e-4,
                                   desc="Frozen ImageNet ConvNeXt-Tiny + linear head"),
    "C_resnet50_finetune":    dict(arch="resnet50", lr_head=1e-3, lr_backbone=1e-4, wd=0.01,
                                   desc="ResNet-50 (ImageNet) fully fine-tuned"),
    "D_efficientnet_b0_finetune": dict(arch="efficientnet_b0", lr_head=1e-3, lr_backbone=3e-4, wd=0.01,
                                   desc="EfficientNet-B0 (ImageNet) fully fine-tuned"),
    "E_convnext_tiny_finetune": dict(arch="convnext_tiny", lr_head=1e-3, lr_backbone=1e-4, wd=0.05,
                                   desc="ConvNeXt-Tiny (ImageNet) fully fine-tuned"),
    "F_convnext_tiny_basic_aug": dict(arch="convnext_tiny", aug="basic", lr_head=1e-3, lr_backbone=1e-4, wd=0.05,
                                   desc="Ablation: ConvNeXt-Tiny fine-tuned with basic augmentation only"),
}
print("Experiments:", list(EXPERIMENTS))

# BATCH 3 — Experiments
One cell per experiment. Run them in order; each one saves to Drive and is skipped next time. Rough T4 times for ~5–7k images: A ≈ 10–15 min, B ≈ 3 min, C/E/F ≈ 8–12 min each, D ≈ 6–8 min. The first epoch's time printout tells you the real pace.
If time runs short, **F is the only optional one** — but it's the evidence that the augmentation choices matter.

In [ ]:
# [B3.01] Experiment A_custom_cnn_scratch
run_experiment("A_custom_cnn_scratch");

In [ ]:
# [B3.02] Experiment B_convnext_linear_probe
run_experiment("B_convnext_linear_probe");

In [ ]:
# [B3.03] Experiment C_resnet50_finetune
run_experiment("C_resnet50_finetune");

In [ ]:
# [B3.04] Experiment D_efficientnet_b0_finetune
run_experiment("D_efficientnet_b0_finetune");

In [ ]:
# [B3.05] Experiment E_convnext_tiny_finetune
run_experiment("E_convnext_tiny_finetune");

In [ ]:
# [B3.06] Experiment F_convnext_tiny_basic_aug
run_experiment("F_convnext_tiny_basic_aug");

In [ ]:
# [B3.07] Experiment status
rows_ = []
for n in EXPERIMENTS:
    r_ = {"experiment": n, "status": experiment_status(n)}
    if r_["status"] == "done":
        m_ = json.loads((EXP_ROOT / n / "metrics.json").read_text(encoding="utf-8"))
        r_.update(test_acc=round(m_["test_acc"], 4), macro_f1=round(m_["test_macro_f1"], 4),
                  train_min=round(m_["train_min"], 1), epochs=m_["epochs_run"], batch_size=m_["batch_size"])
    elif r_["status"] == "failed":
        r_["error"] = (EXP_ROOT / n / "FAILED.txt").read_text(encoding="utf-8").strip().splitlines()[-1][:120]
    rows_.append(r_)
exp_status = pd.DataFrame(rows_); exp_status.to_csv(OUT / "experiments_status.csv", index=False)
display(exp_status)
pending = exp_status[exp_status.status != "done"]
if len(pending):
    print(f"\u26a0\ufe0f {len(pending)} experiment(s) not finished: {pending.experiment.tolist()}. "
          "Re-run the notebook: finished work is kept and unfinished experiments resume. Batch 4 below uses the finished ones.")
else:
    print("\u2705 All experiments finished.")

# BATCH 4 — Analysis & report material (≈2 min)
**Model selection uses the validation set, not the test set.** The test set is used once, for reporting — choosing the winner by test accuracy would make the reported number optimistically biased.

In [ ]:
# [B4.01] Main comparison table with 95% bootstrap confidence intervals
done = [n for n in EXPERIMENTS if (EXP_ROOT / n / "metrics.json").exists()]
if not done:
    raise RuntimeError("No finished experiments yet — see experiments_status.csv and re-run the notebook.")
M = {n: json.loads((EXP_ROOT / n / "metrics.json").read_text(encoding="utf-8")) for n in done}
P = {n: np.load(EXP_ROOT / n / "test_probs.npz") for n in done}
y_test = test_df.label.values
for n in list(done):                                   # safety: only analyse results computed on THIS test split
    if not np.array_equal(P[n]["y"], y_test) or M[n].get("data_sig") != SPLIT_SIG:
        print(f"\u26a0\ufe0f {n} was evaluated on a different split — excluded from the analysis"); done.remove(n)
M = {n: M[n] for n in done}; P = {n: P[n] for n in done}
hard = ~test_df.near_dup.values
rng = np.random.default_rng(SEED)
boot_idx = rng.integers(0, len(y_test), size=(2000, len(y_test)))

rows = []
for n in done:
    correct = (P[n]["clean"].argmax(1) == y_test).astype(float)
    lo, hi = np.percentile(correct[boot_idx].mean(1), [2.5, 97.5])
    m = M[n]
    rows.append({"experiment": n, "approach": m["desc"],
                 "val_macro_F1": m["best_val_macro_f1"],
                 "test_acc": m["test_acc"], "95% CI": f"[{lo:.3f}, {hi:.3f}]",
                 "test_macro_F1": m["test_macro_f1"],
                 "test_acc_hard_subset": correct[hard].mean() if hard.any() else np.nan,
                 "test_acc_TTA": m["test_acc_tta"],
                 "params_M": m["params_M"], "trainable_M": m["trainable_M"],
                 "train_min": m["train_min"], "ms_per_img": m["ms_per_img"], "best_epoch": m["best_epoch"]})
summary = pd.DataFrame(rows).sort_values("val_macro_F1", ascending=False).reset_index(drop=True)
summary.to_csv(OUT / "model_comparison.csv", index=False)

# Selection rule: best validation macro-F1; ties (within 0.5 pt) go to the smaller model
top = summary.val_macro_F1.max()
BEST = summary[summary.val_macro_F1 >= top - 0.005].sort_values("params_M").experiment.iloc[0]
BEST_ARCH = M[BEST]["arch"]
pd.set_option("display.width", 250)
display(summary.round(4))
print(f"\n🏆 Selected model (by validation macro-F1): {BEST}")

In [ ]:
# [B4.02] Selection refinement on VALIDATION data + fair inference-speed benchmark (test set not used)
# Why: at ~99% clean validation F1 the top models differ by only 1–3 validation images, so clean validation F1
# cannot separate them. Among the models within 0.5 pt of the best validation F1 (the tie window), we choose the
# one that is most robust on PERTURBED VALIDATION images (same four perturbations as the robustness test).
# Model size is the final tie-breaker. The test set plays no part in this choice.
top_val = summary.val_macro_F1.max()
cands = summary[summary.val_macro_F1 >= top_val - 0.005].experiment.tolist()
RULE_BEST = summary[summary.experiment.isin(cands)].sort_values("params_M").experiment.iloc[0]   # size-only tie-break
y_val_sel = val_df.label.values

def _load(n):
    mdl, _ = build_model(M[n]["arch"], NUM_CLASSES)
    mdl.load_state_dict(torch.load(EXP_ROOT / n / "best.pt", map_location="cpu"))
    return mdl.to(device, memory_format=torch.channels_last).eval()

vr = []
for n in cands:
    cache = EXP_ROOT / n / "val_robustness.json"
    if cache.exists():
        r = json.loads(cache.read_text(encoding="utf-8"))
    else:
        mdl = _load(n)
        r = {k: float(accuracy_score(y_val_sel, predict_probs(mdl, make_loader(val_df, eval_tf, perturb_kind=k)).argmax(1)))
             for k in ROBUSTNESS_KINDS}
        atomic_write(cache, json.dumps(r, indent=2)); del mdl; gc.collect(); torch.cuda.empty_cache()
    vr.append({"experiment": n, "val_macro_F1": M[n]["best_val_macro_f1"],
               **{f"val_{k}": r[k] for k in ROBUSTNESS_KINDS},
               "val_robust_mean": float(np.mean([r[k] for k in ROBUSTNESS_KINDS])), "params_M": M[n]["params_M"]})
val_rob = (pd.DataFrame(vr).sort_values(["val_robust_mean", "params_M"], ascending=[False, True]).reset_index(drop=True))
val_rob.to_csv(OUT / "selection_validation_robustness.csv", index=False)
display(val_rob.round(4))

BEST = val_rob.experiment.iloc[0]; BEST_ARCH = M[BEST]["arch"]
SELECTION_NOTE = (f"Selection rule: among models within 0.5 pt of the best validation macro-F1 ({', '.join(cands)}), "
                  f"choose the highest mean accuracy on perturbed VALIDATION images (size as final tie-break). "
                  f"Selected: {BEST}. (Size alone would have chosen {RULE_BEST}.) The test set was not used for selection.")
print("\n" + SELECTION_NOTE)

# Fair speed benchmark: same batch size, GPU warmed up, synchronised timing (the training-time number can be
# distorted by other programs or first-call cuDNN tuning)
speed = {}
xb = next(iter(make_loader(test_df, eval_tf)))[0]
xb = to_dev(xb)
for n in done:
    mdl = _load(n)
    with torch.no_grad(), torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
        for _ in range(5): mdl(xb)                                  # warm-up
        if USE_AMP: torch.cuda.synchronize()
        t0 = time.time()
        for _ in range(20): mdl(xb)
        if USE_AMP: torch.cuda.synchronize()
    speed[n] = (time.time() - t0) / (20 * len(xb)) * 1000
    del mdl; gc.collect(); torch.cuda.empty_cache()
summary["ms_per_img"] = summary.experiment.map(speed)
summary.to_csv(OUT / "model_comparison.csv", index=False)
print("\nInference speed (ms/image, warm GPU, batch 32):", {k: round(v, 2) for k, v in speed.items()})
print(f"\n\U0001F3C6 Selected model: {BEST}")

In [ ]:
# [B4.03] Is the winner significantly better? McNemar's exact test on paired test predictions
best_ok = P[BEST]["clean"].argmax(1) == y_test
sig_rows = []
for n in done:
    if n == BEST: continue
    other_ok = P[n]["clean"].argmax(1) == y_test
    b, c = int((best_ok & ~other_ok).sum()), int((~best_ok & other_ok).sum())
    pval = binomtest(b, b + c, 0.5).pvalue if b + c > 0 else 1.0
    sig_rows.append({"vs": n, "only_best_right": b, "only_other_right": c, "p_value": pval,
                     "significant (p<0.05)": pval < 0.05})
sig = pd.DataFrame(sig_rows); sig.to_csv(OUT / "significance_mcnemar.csv", index=False)
display(sig.round(4))

In [ ]:
# [B4.04] Robustness: accuracy under simulated small / cut-off / occluded / low-quality conditions
kinds = ["clean"] + ROBUSTNESS_KINDS
rob = pd.DataFrame({n: {k: accuracy_score(y_test, P[n][k].argmax(1)) for k in kinds} for n in done}).T
rob["mean_drop"] = rob["clean"] - rob[ROBUSTNESS_KINDS].mean(1)
rob.to_csv(OUT / "robustness.csv")
display(rob.round(4))

ax = rob[kinds].plot.bar(figsize=(12, 4.5), width=0.8)
ax.set_ylabel("test accuracy"); ax.set_ylim(0, 1); ax.set_title("Robustness to real-world photo problems")
ax.legend(ncol=5, fontsize=8, loc="lower right"); ax.grid(axis="y", alpha=.3)
plt.xticks(rotation=20, ha="right"); plt.tight_layout(); plt.savefig(FIG / "robustness.png", dpi=150); plt.show()

In [ ]:
# [B4.05] Training curves
H = {n: pd.read_csv(EXP_ROOT / n / "history.csv") for n in done}
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
for n, h in H.items():
    axes[0].plot(h.epoch, h.val_macro_f1, label=n)
    axes[1].plot(h.epoch, h.train_loss, label=n)
    axes[2].plot(h.epoch, h.val_loss, label=n)
for ax, t in zip(axes, ["Validation macro-F1", "Train loss (label-smoothed)", "Validation loss"]):
    ax.set_title(t); ax.set_xlabel("epoch"); ax.grid(alpha=.3)
axes[0].legend(fontsize=7); plt.tight_layout(); plt.savefig(FIG / "training_curves_all.png", dpi=150); plt.show()

fig, axes = plt.subplots(1, len(H), figsize=(4.2 * len(H), 3.4), squeeze=False)
for ax, (n, h) in zip(axes[0], H.items()):
    ax.plot(h.epoch, h.train_acc, label="train acc"); ax.plot(h.epoch, h.val_acc, label="val acc")
    ax.axvline(M[n]["best_epoch"], c="gray", ls="--", lw=1); ax.set_title(n, fontsize=8); ax.set_ylim(0, 1.02)
    ax.grid(alpha=.3); ax.legend(fontsize=7)
plt.tight_layout(); plt.savefig(FIG / "train_vs_val_accuracy.png", dpi=150); plt.show()

In [ ]:
# [B4.06] Error analysis of the selected model
pb = P[BEST]["clean"]; yp = pb.argmax(1)
cm = confusion_matrix(y_test, yp, labels=range(NUM_CLASSES))
cmn = cm / cm.sum(1, keepdims=True).clip(min=1)
sz = max(8, NUM_CLASSES * 0.22); fs = max(4, min(9, 360 // NUM_CLASSES))
plt.figure(figsize=(sz, sz)); plt.imshow(cmn, cmap="Blues", vmin=0, vmax=1); plt.colorbar(fraction=0.046)
plt.xticks(range(NUM_CLASSES), class_names, rotation=90, fontsize=fs); plt.yticks(range(NUM_CLASSES), class_names, fontsize=fs)
plt.xlabel("Predicted"); plt.ylabel("True"); plt.title(f"Normalized confusion matrix — {BEST}")
plt.tight_layout(); plt.savefig(FIG / "confusion_matrix_best.png", dpi=200); plt.show()

off = cm.copy(); np.fill_diagonal(off, 0)
pairs = [(class_names[i], class_names[j], int(off[i, j]))
         for i, j in zip(*np.unravel_index(np.argsort(off, axis=None)[::-1], off.shape)) if off[i, j] > 0][:10]
print("Most confused (true → predicted):"); [print(f"  {t} → {p}: {c}") for t, p, c in pairs]

rep = pd.DataFrame(classification_report(y_test, yp, labels=range(NUM_CLASSES), target_names=class_names,
                                         output_dict=True, zero_division=0)).T.iloc[:NUM_CLASSES]
rep.sort_values("f1-score").to_csv(OUT / "per_class_best.csv")
print("\nWeakest classes:"); display(rep.sort_values("f1-score").head(8).round(3))

wrong = np.where(yp != y_test)[0]
wrong = wrong[np.argsort(-pb[wrong].max(1))][:18]           # most confident mistakes first
if len(wrong):
    fig, axes = plt.subplots(3, 6, figsize=(15, 8.5))
    for ax in axes.ravel(): ax.axis("off")
    for ax, i in zip(axes.ravel(), wrong):
        ax.imshow(IMAGES[test_df.row[i]])
        ax.set_title(f"true {class_names[y_test[i]]}\npred {class_names[yp[i]]} ({pb[i].max():.2f})", fontsize=7, color="red")
    plt.suptitle(f"Misclassified test images — {BEST}"); plt.tight_layout()
    plt.savefig(FIG / "misclassified_best.png", dpi=150); plt.show()

In [ ]:
# [B4.07] Grad-CAM: does the model look at the object or at the background?
def gradcam(model, layer, x):
    store = {}
    h = layer.register_forward_hook(lambda m, i, o: store.__setitem__("a", o))
    x = x.clone().requires_grad_(True)          # makes Grad-CAM work even when backbone weights are frozen
    with torch.enable_grad():
        out = model(x)
        cls = out.argmax(1)
        g = torch.autograd.grad(out.gather(1, cls[:, None]).sum(), store["a"])[0]
    h.remove()
    a = store["a"].detach()
    cam = F.relu((g.mean(dim=(2, 3), keepdim=True) * a).sum(1, keepdim=True))
    cam = F.interpolate(cam, size=x.shape[-2:], mode="bilinear", align_corners=False)[:, 0]
    cam = cam - cam.amin(dim=(1, 2), keepdim=True)
    cam = cam / cam.amax(dim=(1, 2), keepdim=True).clamp_min(1e-8)
    return cam.cpu().numpy(), cls.cpu().numpy()

model, _ = build_model(BEST_ARCH, NUM_CLASSES)
model.load_state_dict(torch.load(EXP_ROOT / BEST / "best.pt", map_location="cpu"))
model = model.to(device, memory_format=torch.channels_last).eval()

r_ = np.random.RandomState(SEED)
right_idx = np.where(yp == y_test)[0]; wrong_idx = np.where(yp != y_test)[0]
pick = list(r_.choice(right_idx, min(8, len(right_idx)), replace=False)) + list(wrong_idx[:4])
x = torch.stack([eval_tf(Image.fromarray(IMAGES[test_df.row[i]])) for i in pick])
cams, preds = gradcam(model, GRADCAM_LAYER[BEST_ARCH](model), to_dev(x))

fig, axes = plt.subplots(2, 6, figsize=(16, 6))
for ax, i, cam, pr in zip(axes.ravel(), pick, cams, preds):
    ax.imshow(IMAGES[test_df.row[i]]); ax.imshow(cam, cmap="jet", alpha=0.45); ax.axis("off")
    ok_ = pr == y_test[i]
    ax.set_title(f"true {class_names[y_test[i]]}\npred {class_names[pr]}", fontsize=8, color="green" if ok_ else "red")
plt.suptitle(f"Grad-CAM — {BEST} (red = regions driving the prediction)"); plt.tight_layout()
plt.savefig(FIG / "gradcam_best.png", dpi=150); plt.show()
del model; torch.cuda.empty_cache()

In [ ]:
# [B4.08] Background-shortcut test: what does the model predict on images that contain NO object?
# If it predicts the photographer's Object ID far above chance, it has learned to recognise backgrounds.
if "BG_IMAGES" in globals() and len(BG_IMAGES):
    model, _ = build_model(BEST_ARCH, NUM_CLASSES)
    model.load_state_dict(torch.load(EXP_ROOT / BEST / "best.pt", map_location="cpu"))
    model = model.to(device, memory_format=torch.channels_last).eval()
    pbg = []
    with torch.no_grad():
        for s in range(0, len(BG_IMAGES), 64):
            x = torch.stack([eval_tf(Image.fromarray(a)) for a in BG_IMAGES[s:s + 64]])
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
                pbg.append(model(to_dev(x)).float().softmax(1).cpu())
    pbg = torch.cat(pbg).numpy(); del model; torch.cuda.empty_cache()
    owner_rate, chance = (pbg.argmax(1) == BG_LABELS).mean(), 1 / NUM_CLASSES
    BG_SHORTCUT_LINE = (f"On {len(BG_IMAGES)} excluded background-only images (no object visible), the selected model "
                        f"predicted the photographer's own Object ID {owner_rate:.1%} of the time (chance = {chance:.1%}), "
                        f"mean confidence {pbg.max(1).mean():.2f}. "
                        + ("This indicates substantial reliance on background cues." if owner_rate > 5 * chance
                           else "This suggests limited reliance on background cues."))
    print(BG_SHORTCUT_LINE)
else:
    print("No background-only images were excluded — shortcut test skipped.")

In [ ]:
# [B4.09] Write a results summary (tables + key facts) to paste into the report
def md_table(d):
    try: return d.to_markdown(index=False)
    except Exception: return "```\n" + d.to_string(index=False) + "\n```"

b = summary.set_index("experiment").loc[BEST]
lines = [
    "# Milestone 1 — Results summary", "",
    "## Dataset",
    f"- {NUM_CLASSES} classes (Object IDs), {len(df_all)} files in the merged dataset → {len(df)} images used.",
    f"- Removed: {N_BG} background-only (filename tag), {N_DUP} byte-identical duplicates, "
    + (f"{N_CAP} by the {CAP_PER_CLASS}-per-class cap, " if N_CAP else "") + f"{N_UNREADABLE} unreadable.",
    f"- Split 70/15/15 stratified per Object ID, seed {SEED}: {len(train_df)} train / {len(val_df)} val / {len(test_df)} test "
    f"(data signature {SPLIT_SIG}).",
    f"- Out-of-fold linear-probe accuracy during the data audit: {(df.oof_pred == df.label).mean():.3f}.",
    f"- Near-duplicate test images (cosine ≥ {DUP_THRESHOLD} to a same-class train image): "
    f"{int(test_df.near_dup.sum())} ({test_df.near_dup.mean():.1%}).", "",
    "## Model comparison (sorted by validation macro-F1)", md_table(summary.round(4)), "",
    "## Robustness (test accuracy under simulated conditions)", md_table(rob.round(4).reset_index(names="experiment")), "",
    "## Significance (McNemar exact test vs. selected model)", md_table(sig.round(4)), "",
    "## Selection on validation data (clean + perturbed)", md_table(val_rob.round(4)), "",
    "## Selected model",
    SELECTION_NOTE,
    f"**{BEST}** — {M[BEST]['desc']}. Validation macro-F1 {b.val_macro_F1:.4f}; "
    f"test accuracy {b.test_acc:.4f} {b['95% CI']}; test macro-F1 {b.test_macro_F1:.4f}; "
    f"hard-subset accuracy {b.test_acc_hard_subset:.4f}; with TTA {b.test_acc_TTA:.4f}; "
    f"{b.params_M:.1f}M parameters; {b.ms_per_img:.2f} ms/image inference (warm GPU, batch 32).",
]
if "BG_SHORTCUT_LINE" in globals():
    lines += ["", "## Background-shortcut test", BG_SHORTCUT_LINE]
(OUT / "results_summary.md").write_text("\n".join(lines), encoding="utf-8")
print("\n".join(lines))
print(f"\nAll figures: {FIG}\nAll tables: {OUT}")

if KEEP_AWAKE and os.name == "nt":
    ctypes.windll.kernel32.SetThreadExecutionState(0x80000000)   # allow sleep again
print("\n\u2705 Notebook finished.")